In [1]:
from my_tokenizer import RegexTokenizer
import torch
import torch.nn as nn 
import torch.nn.functional as F
batch_size=64
bloc_size=256
steps=2000
eval_steps=50
learn_rate=0.0005
device='cuda' if torch.cuda.is_available() else 'cpu'
n_emb=400
n_head=6
n_layers=6
vocab_size = 512
data=open("shakespeare.txt",'r').read()

In [2]:
tokenizer = RegexTokenizer(data)
print("Training...")
tokenizer.train(data,vocab_size)
print("Tokenizer trained successfully!")
encode = lambda ch: tokenizer.encode(ch)
decode = lambda L: tokenizer.decode(L)

Training...
Tokenizer trained successfully!


In [3]:
data=torch.tensor(encode(data),dtype=torch.long)
n=int(len(data)*0.9)
train_data=data[:n]
val_data=data[n:]

In [4]:
torch.manual_seed(42)
def get_batch(split):
    if split=="train":
        data=train_data
    else:
        data=val_data
    ix=torch.randint(len(data)-bloc_size,(batch_size,))
    x=torch.stack([data[i:i+bloc_size]for i in ix])
    y=torch.stack([data[i+1:i+bloc_size+1]for i in ix])
    x,y=x.to(device),y.to(device)
    return x,y
@torch.no_grad()
def estiamte_loss():
    out={}
    model.eval
    for split in ['train','val']:
        losses=torch.zeros(eval_steps)
        for k in range(eval_steps):
            X,Y=get_batch(split)
            logits,loss=model(X,Y)
            losses[k]=loss
        out[split]=losses.mean()
    print("train_loss={} ,validation_loss={}".format(out['train'],out['val']))

In [5]:
torch.manual_seed(42)
class Feed(nn.Module):
    def __init__(self,n_emb):
        super().__init__()
        self.net=nn.Sequential(nn.Linear(n_emb,n_emb),nn.ReLU(),nn.Linear(n_emb,n_emb))
    def forward(self,x):
        return self.net(x)
class Head(nn.Module):
    def __init__(self,head_size):
        super().__init__()
        self.key=nn.Linear(n_emb,head_size,bias=False)
        self.query=nn.Linear(n_emb,head_size,bias=False)
        self.value=nn.Linear(n_emb,head_size,bias=False)
        self.register_buffer('tril',torch.tril(torch.ones(bloc_size,bloc_size)))
    def forward(self,x):
        B,T,C=x.shape
        k=self.key(x)
        q=self.query(x)
        wei=q@k.transpose(-2,-1)*(C**-0.5)
        wei = wei.masked_fill(self.tril[:T,:T] == 0, float('-inf'))
        wei=F.softmax(wei,dim=-1)
        v=self.value(x)
        out=wei@v
        return(out)
class MultiHeadAttention(nn.Module):
    def __init__(self,n_heads,head_size):
        super().__init__()
        self.proj=nn.Linear(n_emb,n_emb)
        self.heads=nn.ModuleList([Head(head_size) for i in range(n_heads)])
    def forward(self,x):
        out=torch.cat([h(x) for h in self.heads],dim=-1)
        out=self.proj(out)
        return out 
class Bloc(nn.Module):
    def __init__(self,n_emb,n_head):
        super().__init__()
        head_size=n_emb//n_head
        self.sa=MultiHeadAttention(n_head,head_size)
        self.feed=Feed(n_emb)
        self.ln1=nn.LayerNorm(n_emb)
        self.ln2=nn.LayerNorm(n_emb)
    def forward(self,x):
        x=x+self.sa(self.ln1(x))
        x=x+self.feed(self.ln2(x))
        return x
class BigramLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.token_embedding_table=nn.Embedding(vocab_size,n_emb)
        self.position_embedding_table=nn.Embedding(bloc_size,n_emb)
        self.blocs=nn.Sequential(*[Bloc(n_emb,n_head=4) for _ in range(n_layers)])
        self.ln_norm=nn.LayerNorm(n_emb)
        self.lm_head=nn.Linear(n_emb,vocab_size)
    def forward(self,idx,targets=None):
        B,T=idx.shape
        token_emb=self.token_embedding_table(idx)
        pos_emb=self.position_embedding_table(torch.arange(T,device=device))
        x=token_emb+pos_emb
        x=self.blocs(x)
        logits=self.lm_head(x)
        if targets==None:
            loss=None
        else:
            A,B,C=logits.shape
            logits=logits.view(B*A,C)
            targets=targets.view(B*A)
            loss=F.cross_entropy(logits,targets)
        return logits,loss
    def generate(self,idx,max_new_tokens):
        for i in range(max_new_tokens):
            idx_cond=idx[:,-bloc_size:]
            logits,loss=self(idx_cond)
            logits=logits[:,-1,:]
            probs=F.softmax(logits,dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx_next = idx_next.to(idx.device)
            idx=torch.cat((idx,idx_next),dim=1)
        return idx
xb,yb=get_batch("train")
model=BigramLanguageModel()
m=model.to(device)
logits,loss=m(xb,yb)

In [ ]:
optimizer=torch.optim.AdamW(m.parameters(),lr=learn_rate)
for step in range (steps):
    xb,yb=get_batch('train')
    logits,loss=m(xb,yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    if step%eval_steps==0:
        print(step," :",end=' ')
        estiamte_loss()

In [ ]:
tokens=encode("my heart is falling")
print(decode(m.generate(idx = torch.tensor(tokens, dtype=torch.long, device=device).unsqueeze(0),max_new_tokens=500)[0].tolist()))